## 1. setup

In [13]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings("ignore")

# ── Paths (same as notebook 06) ───────────────────────────────
BASE_DIR      = Path("/Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project")
FOLDER_VER    = "v3_voltage_random10_issue_flight-removed"
FILE_VER      = "v4_voltage_random10_cleaned"
DATA_DIR      = BASE_DIR / f"model_ready_no_dr_{FOLDER_VER}"
CNN_SCORE_DIR = BASE_DIR / "model_outputs" / f"cnn_lstm_ae_anomaly_score_{FOLDER_VER}"
CKPT_ROOT     = BASE_DIR / "model_outputs" / "07_10split_energy"
OUTPUT_DIR    = BASE_DIR / "model_outputs" / "08_model_comparison"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SPLIT         = "split_05_seed_55"
TARGET        = "target_energy_next_60s_wh"
SEQ_LEN       = 30
BATCH_SIZE    = 512

BASE_FEATURES = [
    "battery_status__cell_count", "battery_status__current_a",
    "battery_status__current_filtered_a", "battery_status__discharged_mah",
    "battery_status__remaining", "battery_status__scale",
    "battery_status__voltage_filtered_v", "battery_status__voltage_v",
    "cumulative_energy_wh", "power_w", "time_s",
    "voltage_per_cell", "voltage_filtered_per_cell",
]
FEATURE_SETS = {
    "Base_13":           BASE_FEATURES,
    "Base_13+CNN_Score": BASE_FEATURES + ["anomaly_score_cnn"],
}

device = torch.device(
    "mps" if torch.backends.mps.is_available() else
    "cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ── Load test data ────────────────────────────────────────────
split_dir = DATA_DIR / SPLIT
test_df   = pd.read_parquet(split_dir / f"test_no_dr_scaled_{FILE_VER}_{SPLIT}.parquet")
test_df   = test_df.sort_values(["log_id", "time_s"]).reset_index(drop=True)

cnn_test  = pd.read_parquet(
    CNN_SCORE_DIR / SPLIT / f"test_cnn_lstm_ae_scored_{SPLIT}.parquet",
    columns=["log_id", "time_s", "anomaly_score_cnn"]
).sort_values(["log_id", "time_s"]).reset_index(drop=True)

if len(test_df) == len(cnn_test):
    test_df["anomaly_score_cnn"] = cnn_test["anomaly_score_cnn"].values.astype(np.float32)
else:
    print("WARN: CNN row mismatch — zeros")
    test_df["anomaly_score_cnn"] = np.float32(0.0)

for col in test_df.select_dtypes("float64").columns:
    test_df[col] = test_df[col].astype(np.float32)

y_true_flat = test_df[TARGET].values
print(f"Test rows: {len(test_df):,}")

# ── Model classes ─────────────────────────────────────────────

class LSTMRegressor(nn.Module):
    def __init__(self, input_size, hidden=64):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden, batch_first=True)
        self.head = nn.Sequential(nn.Linear(hidden, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :]).squeeze(-1)

class BiLSTMRegressor(nn.Module):
    def __init__(self, input_size, hidden=64):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden, batch_first=True, bidirectional=True)
        self.head = nn.Sequential(nn.Linear(hidden*2, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :]).squeeze(-1)

def predict_seq(model, df, feat_cols):
    model.eval()
    # keep row order aligned with test_df for flat metric computation
    all_true, all_pred = [], []
    for lid, grp in df.sort_values(["log_id", "time_s"]).groupby("log_id"):
        X = grp[feat_cols].to_numpy(dtype=np.float32)
        y = grp[TARGET].to_numpy(dtype=np.float32)
        n = len(X)
        if n <= SEQ_LEN:
            continue
        seqs = np.stack([X[i-SEQ_LEN:i] for i in range(SEQ_LEN, n)])
        preds = []
        with torch.no_grad():
            for batch in torch.split(torch.tensor(seqs), BATCH_SIZE):
                preds.extend(model(batch.to(device)).cpu().numpy().tolist())
        all_true.extend(y[SEQ_LEN:].tolist())
        all_pred.extend([max(p, 0.0) for p in preds])
    return np.array(all_true), np.array(all_pred)

def metrics(y_true, y_pred):
    return {
        "MAE_Wh":  round(mean_absolute_error(y_true, y_pred), 4),
        "RMSE_Wh": round(np.sqrt(mean_squared_error(y_true, y_pred)), 4),
        "R2":      round(r2_score(y_true, y_pred), 4),
    }

# ── Evaluate LSTM + BiLSTM ────────────────────────────────────
rows = []

for feat_name, feat_cols in FEATURE_SETS.items():
    available = [f for f in feat_cols if f in test_df.columns]
    for arch_name, ModelClass in [("LSTM", LSTMRegressor), ("BiLSTM", BiLSTMRegressor)]:
        ckpt = CKPT_ROOT / f"checkpoints_{SPLIT}" / f"ckpt_{arch_name}_{feat_name}.pt"
        if not ckpt.exists():
            print(f"MISSING: {ckpt.name} — run notebook 07 first")
            continue
        model = ModelClass(input_size=len(available)).to(device)
        model.load_state_dict(torch.load(ckpt, map_location=device, weights_only=True))
        y_true, y_pred = predict_seq(model, test_df, available)
        m = metrics(y_true, y_pred)
        label = arch_name if feat_name == "Base_13" else f"{arch_name}+CNN"
        rows.append({"Model": label, **m})
        print(f"  {label}  MAE={m['MAE_Wh']}  RMSE={m['RMSE_Wh']}  R²={m['R2']}")
        del model

# ── Merge with existing table ─────────────────────────────────
existing = pd.DataFrame([
    {"Model": "Linear Regression",        "MAE_Wh": 0.3208, "RMSE_Wh": 0.7944, "R2": 0.8212},
    {"Model": "XGBoost",                  "MAE_Wh": 0.3241, "RMSE_Wh": 0.8117, "R2": 0.8173},
    {"Model": "Power Persistence Baseline","MAE_Wh": 0.3602, "RMSE_Wh": 0.8609, "R2": 0.7901},
    {"Model": "Random Forest",             "MAE_Wh": 0.3392, "RMSE_Wh": 0.8327, "R2": 0.8074},
    {"Model": "Mean Predictor",            "MAE_Wh": 1.5109, "RMSE_Wh": 1.9738, "R2": -0.0623},
])

new_rows = pd.DataFrame(rows)
combined = pd.concat([existing, new_rows], ignore_index=True)
combined = combined.sort_values("MAE_Wh").reset_index(drop=True)
combined.index += 1
combined.index.name = "Rank"

print("\n" + "="*65)
print("FULL MODEL COMPARISON — Test Set")
print("="*65)
print(combined[["Model","MAE_Wh","RMSE_Wh","R2"]].to_string())

combined.to_csv(OUTPUT_DIR / "08_full_model_comparison.csv")

Device: mps
Test rows: 162,502
  LSTM  MAE=0.6839  RMSE=1.2383  R²=0.7864
  BiLSTM  MAE=0.7707  RMSE=1.7378  R²=0.5793
  LSTM+CNN  MAE=1.2815  RMSE=2.4289  R²=0.1781
  BiLSTM+CNN  MAE=0.6269  RMSE=1.0895  R²=0.8347

FULL MODEL COMPARISON — Test Set
                           Model  MAE_Wh  RMSE_Wh      R2
Rank                                                     
1              Linear Regression  0.3208   0.7944  0.8212
2                        XGBoost  0.3241   0.8117  0.8173
3                  Random Forest  0.3392   0.8327  0.8074
4     Power Persistence Baseline  0.3602   0.8609  0.7901
5                     BiLSTM+CNN  0.6269   1.0895  0.8347
6                           LSTM  0.6839   1.2383  0.7864
7                         BiLSTM  0.7707   1.7378  0.5793
8                       LSTM+CNN  1.2815   2.4289  0.1781
9                 Mean Predictor  1.5109   1.9738 -0.0623


In [1]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path("/Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project")

# Check all result CSVs
for p in sorted(BASE_DIR.rglob("*.csv")):
    if any(k in p.name for k in ["result","summary","comparison","baseline"]):
        print(p)

/Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project/eda_outputs/01_dataset_summary.csv
/Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project/eda_outputs/01_final_dataset_summary.csv
/Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project/investigation_outputs/abnormal_verdict_summary.csv
/Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project/investigation_outputs/flight_removal_v5/label_balance_summary.csv
/Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project/investigation_outputs/flight_removal_v5/suspicious_voltage_summary.csv
/Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project/model_outputs/04_baseline_models/all_splits_raw_results.csv
/Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project/model_outputs/04_baseline_models/baseline_10split_summary.csv
/Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project/model_outputs/04_baseline_models/baseline_results_by_label.csv